In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import preliz as pz
import pymc as pm
import pytensor.tensor as pt

from patsy import dmatrix

import pymc_extras.statespace as pmss

from pymc_extras.statespace import structural as st

plt.rcParams.update(
    {
        "figure.figsize": (14, 4),
        "figure.dpi": 144,
        "figure.constrained_layout.use": True,
        "axes.grid": True,
        "grid.linewidth": 0.5,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.bottom": False,
        "axes.spines.left": False,
        "axes.spines.right": False,
    }
)

In [ ]:
def compile_component(component):
    """Helper function to call compile_statespace directly on Components"""
    return pmss.compile_statespace(component.build(verbose=False))

In [ ]:
seed = sum(map(ord, "Structural Timeseries"))

rng = np.random.default_rng(seed)

# Introduction

This notebook is an introduction to the structural timeseries modeling framework introduced in `pymc_extras.statespace`.

The structural timeseries framework, described by Durbin and Koopsman (2010) in Chapter 3.6, is a way to decompose a univariate timeseries into component parts:

$$y_t = \mu_t + \gamma_t + c_t + \varepsilon_t$$

Where $\mu_t$ is a trend component, $\gamma_t$ is a seasonal component, $c_t$ is a cycle component, and $\varepsilon_t$ is a measurement error. Each of the components can themselves be constant or have stochastic innovations, rendering them time-varying.

The API for this class of model is modular, and allows you to mix-and-match components to make the time series of your dreams.

# Level and Trend Component

The most basic componet is the `LevelTrend` component, represented by $\mu_t$. This component allows for a time trend wit as many time derivatives as you like. To illustrate, we begin with a constant mean model.

`LevelTrend` has two argumnets: `order` and `innovations_order`. `order` controls the number of non-zero time derivatives to include in the trend, while `innovations_order` is the number of components to have a stochastic innovation. To make a constant level, we include 1 derivative, and no innovations

In [ ]:
ll = st.LevelTrend(order=1, innovations_order=0)
param_dict = {"initial_level_trend": np.array([1])}

f = compile_component(ll)
plt.plot(f(**param_dict, steps=100, draws=1)[1].T)

If we have 2 derivatives with no innovations, we get a deterministic trend model

In [ ]:
ll = st.LevelTrend(order=2, innovations_order=0)
f = compile_component(ll)
param_dict = {"initial_level_trend": np.array([1, 0.25])}
plt.plot(f(**param_dict, steps=100, draws=1)[1])

With 3 we can have a quadratic trend, and so on

In [ ]:
ll = st.LevelTrend(order=3, innovations_order=0)
f = compile_component(ll)

param_dict = {"initial_level_trend": np.array([1, 0.25, 0.01])}
plt.plot(f(**param_dict, steps=100)[1].T)

To introduce time-varying elements into the model, we set the innovations order.

* By default, the innovations order is equal to the order (all components of the trend are time-varying)
* If you pass an integer, all components beyond that integer will be static
* You can also pass a list of `1` for "include innovations at this order" and `0` for "exclude at this order"

A Gaussian Random Walk can be implemented using `order=1, innovations_order=1`

In [ ]:
grw = st.LevelTrend(order=1, innovations_order=1)
f = compile_component(grw)

param_dict = {"initial_level_trend": np.ones((1,)), "sigma_level_trend": np.array([0.25])}

plt.plot(f(**param_dict, steps=100, draws=10)[1].T)

A random walk with drift would have `order=2` but `innovations_order = 1`

In [ ]:
rwd = st.LevelTrend(order=2, innovations_order=1)
f = compile_component(rwd)

param_dict = {"initial_level_trend": np.array([0, -0.25]), "sigma_level_trend": np.array([0.25])}
plt.plot(f(**param_dict, steps=100, draws=10)[1].T)

The local level model has `order=2` and `innovations_order = 2`. It corresponds to a Random walk with drift, where the drift also follows a random walk.

This is the first model where it's interesting to look at both hidden states and see their contributions to the final model. As promised, the trend component follows a random walk, while the level is a random walk with drift, with the drift at time $t$ equal to the trend at time $t-1$.

In [ ]:
ll = st.LevelTrend(order=2)
param_dict = {"initial_level_trend": np.zeros((2,)), "sigma_level_trend": np.array([0.25, 1e-4])}
f = compile_component(ll)

fig, ax = plt.subplots(1, 2)
xs, ys = f(**param_dict, draws=10, steps=100)
titles = ["Level", "Trend"]
for i, axis in enumerate(fig.axes):
    axis.plot(xs[..., i].T)
    axis.set_title(titles[i])
plt.show()

Durbin and Koopsman mention that many analysts find the level components produced by the local level model change too rapidly. In this case, they suggest to disable the innovations on the level, and have them only on the trend:

In [ ]:
smooth_level = st.LevelTrend(order=2, innovations_order=[0, 1])
param_dict = {"initial_level_trend": np.zeros((2,)), "sigma_level_trend": np.array([0.25])}
f = compile_component(smooth_level)

fig, ax = plt.subplots(1, 2)
xs, ys = f(**param_dict, draws=10, steps=100)
titles = ["Level", "Trend"]
for i, axis in enumerate(fig.axes):
    axis.plot(xs[..., i].T)
    axis.set_title(titles[i])
plt.show()

# Seasonal Components

Durbin and Koopsman present argue for two "cannonical" seasonal components: on in the time domain, and one in the frequency domain. Both are available in `statespace.structural`, as `TimeSeasonality` and `FrequencySeasonality`, respectively.

## Time Domain Seasonality

Time domain seasonality is autoregressive. Seasonal effects are a repeating pattern over `season_length`, with the net effect summing to 0.

In [ ]:
monthly_season = st.TimeSeasonality(name="annual_seasonality", season_length=12, innovations=False)
param_dict = {"params_annual_seasonality": rng.normal(size=11)}
f = compile_component(monthly_season)

x, y = f(**param_dict, steps=12 * 12)
plt.plot(y.T)

As promised, they sum to zero

In [ ]:
np.allclose(y[1:].reshape(-1, 12).sum(axis=1), 0)

When `innovations = True`, the seasonal components are time varying.

In [ ]:
monthly_season = st.TimeSeasonality(name="annual_seasonality", season_length=12, innovations=True)
param_dict = {
    "params_annual_seasonality": np.zeros(
        11,
    ),
    "sigma_annual_seasonality": 0.25,
}
f = compile_component(monthly_season)

xs, ys = f(**param_dict, steps=12 * 12, draws=10)
plt.plot(ys.T)

## Frequency Domain Seasonality

An alternative to using lagged terms is to use Fourier terms. In this case, the user can specify both the `seasonal_length` and the number of fourier terms to include. At `seasonal_length // 2` the representation is "saturated", so that's the highest you'll ever need to go (and that's what you get by default). Also important: at saturation, the last initial state is not identified. So if you request a saturated model, there will be one less initial parameter than you might expect.

Here are the six different frequencies we get for a monthly model. Since there are no innnovations, the strength of each frequency is entirely determined by the initial state. As we notes above, we only need to specify 11 initial states instead of 12, because the model is fully saturated.

In [ ]:
monthly_season = st.FrequencySeasonality(
    name="monthly_seasonality", season_length=12, n=6, innovations=False
)
param_dict = {
    "params_monthly_seasonality": np.ones(
        11,
    )
}
f = compile_component(monthly_season)
x, y = f(**param_dict, steps=12 * 12)
fig, ax = plt.subplots(3, 2)
for i, axis in enumerate(fig.axes):
    axis.plot(x[..., i])

Each frequency is comprised of one observed and one hidden component. The total seasonal pattern is the sum of the observed states (the even ones)

In [ ]:
monthly_season.ssm["design"].eval()

In [ ]:
fig, ax = plt.subplots()
ax.plot(y)

When `innovations=True`, the states are time-varying

In [ ]:
monthly_season = st.FrequencySeasonality(
    name="monthly_seasonality", season_length=12, n=6, innovations=True
)
f = compile_component(monthly_season)

param_dict = {
    "params_monthly_seasonality": np.ones(
        11,
    ),
    "sigma_monthly_seasonality": 0.07,
}
xs, ys = f(**param_dict, steps=12 * 12, draws=10)
fig, ax = plt.subplots(3, 2)
for i, axis in enumerate(fig.axes):
    axis.plot(xs[..., i].T)

## Autoregressive Component

The autoregressive component can be thought of as a way to introduce serially correlated errors into the model, rather than just white noise. The `order` argument governs how many lags to introduce. Like the `innovations_order` argumnet of the `LevelTrend`, the `AutoregressiveComponent` can also take a list of 1's and 0's to specify which AR orders you would like included.

With no innovations, the AR component will look like an impulse response function

In [ ]:
ar = st.Autoregressive(name="ar", order=1)
f = compile_component(ar)
param_dict = {"params_ar": np.array([0.95]), "sigma_ar": 0.01}
x, y = f(**param_dict, steps=100)
plt.plot(y)

One has to take care with the AR parameters, however. If they aren't stationary, the model won't be etiher. Ideally, non-stationary behavior should come from the level components, not the AR components. Actually, an AR(1) with `rho = 1` is just a Gaussian random walk!

If the parameter is bigger than one in absolutely value, you will get exponential blowup

In [ ]:
param_dict = {"params_ar": np.array([1.1]), "sigma_ar": 0.01}
x, y = f(**param_dict, steps=100)
plt.plot(y)

Higher order AR processes are more difficult to keep stationary. Nevertheless, here's draws form an AR(3) with innovations

In [ ]:
ar3 = st.Autoregressive(name="ar", order=3)
f = compile_component(ar3)
param_dict = {"params_ar": np.array([0.7, -0.25, 0.03]), "sigma_ar": 0.1}
xs, ys = f(**param_dict, steps=100, draws=100)
plt.plot(ys.T)

## Cycle Component

A cycle is defined in [Forecasting: Principals and Practice](https://otexts.com/fpp3/components.html) as a long-term pattern of peaks and valleys with an unpredictable, changing duration. This module follows Durbin and Koopsman in taking a more narrow definition. Here, a cycle is simply:

 1. A long frequency seasonal component, lasting between 1.5 and 13 years, with;
 2. Only one fourier feature, so that it is modeled as pure sine wave, and;
 3. May include a dampening term, to induce stationarity in the presence of time-variation

Specifically, the `Cycle` component is defined as:

$$\begin{align}
    \begin{bmatrix} c_{t+1} \\ c_{t+1}^\star \end{bmatrix} &= \rho_c \begin{bmatrix} \cos \lambda_c & \sin \lambda_c \\ -\sin \lambda_c & \cos \lambda_c \end{bmatrix} + \begin{bmatrix}\tilde \omega_t \\ \tilde \omega_t^\star \end{bmatrix}, & \begin{bmatrix} \omega_t \\ \tilde \omega_t^\star \end{bmatrix} &\sim N \left (0, \begin{bmatrix} \sigma_c^2 & 0 \\ 0 & \sigma_c^2 \end{bmatrix} \right ) \\
    c_t &= \begin{bmatrix} 1 & 0 \end{bmatrix} \begin{bmatrix} c_t \\ c_t^\star \end{bmatrix}
    \end{align}$$

Where $\rho_c$ is the dampening factor, and $\lambda_c$ is the period. Unlike the seasonal component, it is recommended to estimate $\lambda_c$, rather than setting it to a fixed value. Nevertheless, it is still possible to set it to a fixed value.

This first example sets the cycle to a fixed value. In this case it is identically equal to `FrequencySeasonal(s=13, n=1)`

In [ ]:
cycle = st.Cycle(name="cycle", cycle_length=13, innovations=False)
f = compile_component(cycle)
param_dict = {"params_cycle": np.array([1.0, 0])}
x, y = f(**param_dict, steps=100)
plt.plot(y)

Unique contributions come when we allow the period length to be estimated, and when a dampening term is present.

In [ ]:
cycle = st.Cycle(name="cycle", estimate_cycle_length=True, innovations=False, dampen=True)
f = compile_component(cycle)
param_dict = {
    "params_cycle": np.array([1.1, 0.0]),
    "length_cycle": rng.uniform(1.5, 13),
    "dampening_factor_cycle": 0.95,
}
fig, ax = plt.subplots()
for _ in range(10):
    param_dict["length_cycle"] = rng.uniform(1.5, 13)
    x, y = f(**param_dict, steps=100)
    ax.plot(y)
plt.show()

When paired with hidden state innovations, this results in long-term stationary behavior. The dampening term prevents the cycle from growing larger and larger over time.

You might argue that none of this looks particularly "cyclical", and you might be right. But this provides a tractable representation to an otherwise difficult component to capture.

In [ ]:
cycle = st.Cycle(name="cycle", estimate_cycle_length=True, innovations=True, dampen=True)
f = compile_component(cycle)
param_dict = {
    "params_cycle": np.array([1.1, 0.0]),
    "dampening_factor_cycle": 0.95,
    "sigma_cycle": 0.1,
}
fig, ax = plt.subplots()
for _ in range(10):
    param_dict["length_cycle"] = rng.uniform(1.5, 13)
    x, y = f(**param_dict, steps=100)
    ax.plot(y)
plt.show()

## Exogenous Regression

One last component is the `Regression` component. This allows you to introduce exogenous data into your structural model. The way this is done is a bit unusal: the regression coefficents are introduced as hidden states, and the exogenous data is appended to the design matrix. The state space matrices look like this:

$$\begin{align}
T &= \begin{bmatrix} 1 & 0       & \dots & 0 \\
                    0       & 1 & \dots  & 0 \\
                    \vdots  & \vdots  & \ddots & 0 \\
                    0       &       0 & \dots  & 1
\end{bmatrix}
& Z_t &= \begin{bmatrix} A_{t, 1} & A_{t, 2} & \dots & A_{t, 3} \end{bmatrix}
& X_0 &= \begin{bmatrix} \beta_1 \\ \beta_2 \\ \vdots \\ \beta_k \end{bmatrix}
\end{align}$$

Where $A$ is a matrix of exogenous data. So $Z$ becomes time-varying, and contains the data. Suppose there are no state innovations, then:

$$\begin{align}
    X_t &= T X_{t-1} \\
    y_t &= Z_t X_t \end{align}
$$
Since $T$ is the identity matrix, $X_0$ is a column vector of regression coeffecients, and $Z_t$ is the rows of the data, $y_t$ ends up as the regression $y_t = \sum_{i=0}^k \beta_i \cdot a_{t,i}$. This is completely equivalent to OLS, just computed recursively.

Here is an example of an only complicated OLS regression

In [ ]:
def plot_exog_regression(x, y, data):
    fig = plt.figure(figsize=(14, 6))
    gs = plt.GridSpec(nrows=2, ncols=2, figure=fig)

    y_axis = fig.add_subplot(gs[0, :])
    data_axis = fig.add_subplot(gs[1, 0])
    param_axis = fig.add_subplot(gs[1, 1])
    axes = [y_axis, data_axis, param_axis]
    datas = [y, data, x]
    titles = ["Observed State", "Hidden States (Data)", "Regression Coefficents"]
    for axis, data, title in zip(axes, datas, titles):
        axis.plot(data)
        axis.set_title(title)
    plt.show()

In [ ]:
reg = st.Regression(name="exog", state_names=["x1", "x2", "x3"], innovations=False)
f = compile_component(reg)
exog_data = rng.normal(size=(100, 3))
true_betas = rng.normal(size=(3,))
param_dict = {"beta_exog": true_betas, "data_exog": exog_data}

x, y = f(**param_dict, steps=100)
plot_exog_regression(x, y, exog_data)

As promised, every state in `y` is the regression `x0 @ data`

In [ ]:
np.allclose(y[1:], np.einsum("nk, k -> n", exog_data, true_betas))

Another way to use the `Regression` is with innovations. In this case, the regression coefficients will follow a random walk, $\beta_t = \beta_{t+1} + \varepsilon_t$, and the regression will be time-varying.

In the next example, a time-varying regression is combined with spines to make convincing cyclical patterns. All spines are initiall zero, and cycle-like behavior emerges as the different spline components become weaker/stronger over time.

In [ ]:
time = np.arange(100)
n_knots = 10
knot_points = np.quantile(time, np.linspace(0, 1, n_knots))[1:-1]

B = dmatrix(
    "bs(year, knots=knots, degree=3, include_intercept=True) - 1",
    {"year": time, "knots": knot_points},
)
reg = st.Regression(name="exog", state_names=[f"x{i}" for i in range(B.shape[1])], innovations=True)
f = compile_component(reg)

param_dict = {
    "beta_exog": np.zeros(reg.k_states),
    "data_exog": np.asarray(B),
    "sigma_beta_exog": rng.normal(scale=0.025, size=(reg.k_states)) ** 2,
}
x, y = f(**param_dict, steps=100)

plot_exog_regression(x, y, B)

# Combining components

More interesting models can be created by combining components. Components are combined by simple addition.

There is one special component, the `MeasurementError` component, that cannot be used on its own. It adds measurement error to the model.

Here is a local-linear trend model with monthly seasonality and measurement error:

In [ ]:
ll = st.LevelTrend(order=2)
monthly_season = st.TimeSeasonality(name="monthly_seasonality", season_length=12, innovations=False)
measurement_error = st.MeasurementError(name="obs")

param_dict = {
    "initial_level_trend": np.zeros(
        2,
    ),
    "params_monthly_seasonality": rng.normal(size=11),
    "sigma_level_trend": np.array([0.25, 0.01]),
    "sigma_obs": 0.1,
}

mod = ll + monthly_season + measurement_error
f = compile_component(mod)

x, y = f(**param_dict, steps=100)
plt.plot(y)

Nothing stops you from having many of the same component. This is especially relevant for seasonal components. Here is a model with both annual and monthly seasonality, in addition to the smooth trend

In [ ]:
n_month = 1
n_year = 1

smooth_trend = st.LevelTrend(order=2, innovations_order=[0, 1])
ar = st.Autoregressive(name="ar", order=1)
monthly_season = st.FrequencySeasonality(
    name="monthly_seasonality", season_length=30, n=n_month, innovations=False
)
annual_season = st.FrequencySeasonality(
    name="annual_seasonality", season_length=365, n=n_year, innovations=False
)

param_dict = {
    "initial_level_trend": np.array([100, 0]),
    "sigma_level_trend": np.array([1e-2]),
    "params_monthly_seasonality": rng.normal(scale=20, size=n_month * 2),
    "params_annual_seasonality": rng.normal(scale=20, size=n_year * 2),
    "params_ar": np.array([0.95]),
    "sigma_ar": 10,
}

mod = smooth_trend + ar + monthly_season + annual_season
f = compile_component(mod)
x, y = f(**param_dict, steps=365 * 3)
plt.plot(y)

We can see the different components by looking at the model's hidden states

In [ ]:
fig, ax = plt.subplots(4, figsize=(14, 9), sharex=True)

plot_indexes = [0, 2, slice(3, 3 + n_month * 2, 2), slice(-(n_year * 2), None, 2)]
sum_state_flags = [False, False, True, True]
titles = ["Trend", "Autoregressive", "Season Length = 30", "Season Length = 365"]
for axis, idx, sum_state, title in zip(fig.axes, plot_indexes, sum_state_flags, titles):
    hidden_state = x[:, idx]
    if sum_state:
        hidden_state = hidden_state.sum(axis=-1)
    axis.plot(hidden_state)
    axis.set(title=title)
plt.show()

# PyMC Integration

Up until now I've been manually setting values in the statespace matrices, which is **not** how this should be used. This has been for illustration purposes only.

What you should actually do is build a `PyMCStateSpace` model using the `.build()` method.

## Nile Data

Let's model the Nile river dataset with a smooth local level model with AR innovations and measurement error

In [ ]:
nile = pd.read_csv("../tests/statespace/_data/nile.csv", dtype={"x": "float64"})
nile.index = pd.date_range(start="1871-01-01", end="1970-01-01", freq="YS-JAN")
nile.rename(columns={"x": "height"}, inplace=True)
nile = (nile - nile.mean()) / nile.std()
nile.plot()

### Model Definition

In [ ]:
mod = st.LevelTrend(order=2, innovations_order=[0, 1])
mod += st.Autoregressive(name="ar", order=1)
mod += st.MeasurementError(name="obs")
ss_mod = mod.build(name="nile")

In [ ]:
initial_trend_dims, sigma_trend_dims, ar_param_dims, P0_dims = ss_mod.param_dims.values()
coords = ss_mod.coords

### Priors and Sampling

Priors are very important. I choose a very very tight Gamma prior on `sigma_trend`, because I want only smooth, gradual changes in the trend. In statespace models, the `sigma` terms control the size of the innovations to the latent states, which in turn dictate how rapidly they are allowed to change. If `sigma_trend` is very large, we will end up with a level term that rapidly changes every year. On the other hand, we set the AR innovations to be relatively large, because we want it to act like a "garbage can", absorbing all the variation in the time series that isn't explained by gradual changes in the level.

Finally, we have observation noise. I set a HalfNormal with a standard deviation of 0.05. Since the data is normalized, this implies that my prior belief is that any observation in the dataset could be wrong by at most something like $\pm 1.96\sigma \approx \pm0.1$ standard deviations. We could choose to omit measurement error from this model, but it's a good practice to include it in general (do we really believe these recorded values are *exactly* the level of the nile in the early 1900s?). It also helps with numerical stability, by adding a bit of noise on the diagonal of the Kalman Gain matrix, ensuring we never end up with degeneracies during estimation (although a small diagonal jitter is quietly added if you omit observation noise to try to dodge this issue)

In [ ]:
with pm.Model(coords=coords) as model_1:
    P0_diag = pm.Gamma("P0_diag", alpha=5, beta=5)
    P0 = pm.Deterministic("P0", pt.eye(ss_mod.k_states) * P0_diag, dims=P0_dims)

    initial_level_trend = pm.Normal("initial_level_trend", dims=initial_trend_dims)
    params_ar = pm.Beta("params_ar", alpha=3, beta=3, dims=ar_param_dims)

    sigma_level_trend = pm.Gamma("sigma_level_trend", alpha=2, beta=50, dims=sigma_trend_dims)
    sigma_ar = pm.Gamma("sigma_ar", alpha=2, beta=5)
    sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.05)

    ss_mod.build_statespace_graph(nile)
    idata = pm.sample()

In [ ]:
az.plot_trace_dist(idata, var_names=["~P0"], filter_vars="like")

In [ ]:
post_pred = ss_mod.sample_conditional_posterior(idata)

We can look at smoother outputs since there is measurement noise. If there is no measurement noise, the smoother and filter will noiselessly encode the data, so only the outputs from the one-step ahead predictions are interesting.

We can start by looking at the different estimated components. There is a helper method, `extract_components_from_idata`, to grab them out of the `idata` returned by post-estimation sampling functions, such as `sample_conditional_posterior`.

### Time Series Decomposition

In [ ]:
component_idata = ss_mod.extract_components_from_idata(post_pred)
component_hdi = az.hdi(component_idata)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 9))

for axis, state in zip(fig.axes, component_idata.coords["state"].values):
    axis.plot(
        nile.index,
        component_idata.stack(sample=["chain", "draw"])
        .smoothed_posterior.sel(state=state)
        .mean(dim="sample"),
        color="tab:orange",
    )
    axis.fill_between(
        nile.index,
        *component_hdi.smoothed_posterior.sel(state=state).values.T,
        color="tab:blue",
        alpha=0.15,
    )
    axis.set_title(state.title())

All structural models in the statespace package are additive. To get the estimated series, just add together the components.

Since we're looking at the smoothed outputs, the mean of the latent states is exactly equal to the data. This is because the smoother gets to look at the observed data at time t when making updates. You could instead plot the `predicted_posterior` to get a picture of how the model is performing on in-sample prediction.


In [ ]:
fig, ax = plt.subplots()
hdi = az.hdi(component_idata)
ax.plot(
    nile.index,
    (
        component_idata.stack(sample=["chain", "draw"])
        .smoothed_posterior.sum(dim="state")
        .mean(dim="sample")
    ),
    color="tab:orange",
    label="Mean smoothed observed",
)
ax.plot(nile.index, nile.values, color="tab:red", lw=2, label="Data", ls="--")
ax.fill_between(
    nile.index,
    *hdi.smoothed_posterior.sum(dim="state").values.T,
    color="tab:blue",
    alpha=0.15,
    label="HDI 94%",
)
ax.legend()
plt.show()

### Forcasting

Forcasts are also possible. The model extrapolates the trend forward, predicting the level of the nile will continue to decrease. This is probably unrealistic -- it would be nice if there was some mean reverting behavior in the trend. But that's another problem for another day!

In [ ]:
nile_forecast = ss_mod.forecast(idata, start=nile.index[-1], periods=10)

In [ ]:
fig, ax = plt.subplots()
ax.plot(nile.index, nile.values)

forecast_time = nile_forecast.coords["time"]
mu_forecast = nile_forecast.forecast_observed.mean(dim=["chain", "draw"]).isel(observed_state=0)
hdi_forecast = az.hdi(nile_forecast.to_dataset().forecast_observed).isel(observed_state=0)

ax.plot(forecast_time, mu_forecast)
ax.fill_between(forecast_time, *hdi_forecast.values.T, alpha=0.25, color="tab:orange")
plt.show()

## Airline Data

Another famous timeseries dataset is the airline dataset. It has both a non-stationary trend and a seasonal component. We can model it with a local level and a seasonal trend. The seasonal trend will be time-varying, because it appears ot be increasing in strength. Unlike the nile model, we will allow innovations to the level term (rather than just the trend), which will result in a more jagged, rapidly changing trend when we decompose the time series.

Finally, we can leave away the measurement error just to emphasize that it's not necessary to have

In [ ]:
airpass = pd.read_csv(
    "../tests/statespace/_data/airpass.csv",
    parse_dates=True,
    date_format="%Y %b",
    index_col=0,
    dtype={"value": float},
).rename(columns={"value": "passengers"})
airpass.index.freq = airpass.index.inferred_freq
airpass.apply(np.log).plot()

### Model Definition

In [ ]:
ll = st.LevelTrend(order=2)
se = st.FrequencySeasonality(season_length=12, name="annual")
me = st.MeasurementError(name="error")
mod = ll + se + me
ss_mod = mod.build()

### Priors and Sampling

In [ ]:
initial_trend_dims, sigma_trend_dims, annual_dims, P0_dims = ss_mod.param_dims.values()
coords = ss_mod.coords

In [ ]:
with pm.Model(coords=coords) as model_2:
    P0_diag = pm.Gamma("P0_diag", alpha=2, beta=1, dims=("state",))
    P0 = pm.Deterministic("P0", pt.diag(P0_diag), dims=("state", "state_aux"))
    initial_trend = pm.Normal("initial_level_trend", sigma=[100, 1], dims=("state_level_trend",))
    annual_seasonal = pm.ZeroSumNormal("params_annual", sigma=100, dims=("state_annual",))

    sigma_trend = pm.Gamma("sigma_level_trend", alpha=2, beta=[1, 10], dims=("shock_level_trend",))
    sigma_monthly_season = pm.Gamma("sigma_annual", alpha=2, beta=1)
    sigma_error = pm.Gamma("sigma_error", alpha=2, beta=1)

    ss_mod.build_statespace_graph(airpass)
    idata = pm.sample()

In [ ]:
az.plot_trace_dist(idata, var_names=["~P0"], filter_vars="like")

### Time Series Decomposition

One helpful argument to know is `mvn_method`. This controls how samples from a multivariate normal are generated. To get samples, we always have to do a matrix decomposition. By default, we use SVD, because it is robust to singular covariance matrices. These almost always arise when statespace models do not have measurement error.

SVD is, however, quite slow. When you have measurement error in the model, you can try to speed up sampling by choosing `mvn_method="cholesky"` instead.

There is also `mvn_method='eigh'`, which lives between `"cholesky"` and `"svd"` in terms of speed and robustness.

In [ ]:
post = ss_mod.sample_conditional_posterior(idata, mvn_method="cholesky")

As before, we compress the raw outputs to the component "states" using `extract_components_from_idata`

In [ ]:
component_idata = ss_mod.extract_components_from_idata(post)
component_states = component_idata.coords["state"].values.tolist()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 9))
x_values = component_idata.coords["time"]
for axis, name in zip(fig.axes, component_states):
    data = component_idata.smoothed_posterior.sel(state=name)
    hdi = az.hdi(data)
    mean = data.mean(dim=["chain", "draw"])

    axis.plot(x_values, mean)
    axis.fill_between(x_values, *hdi.values.T, color="tab:blue", alpha=0.1)
    axis.set_title(name)
plt.show()

### Latent vs Observed States

In the Nile example, we rebuilt the in-sample outputs by adding together the structural components. This is always possible, but it is not necessary. If all you need is the filter outputs corresponding to the observed time series, simply ask for `predicted_posterior_observed`, `filtered_posterior_observed`, or `smoothed_posterior_observed`

In [ ]:
fig, ax = plt.subplots()
x_values = post.coords["time"]
hdi_post = az.hdi(post.to_dataset())

ax.plot(
    x_values,
    post.smoothed_posterior_observed.isel(observed_state=0).mean(dim=["chain", "draw"]),
    label="Posterior Mean, Smoother",
)
ax.fill_between(
    x_values,
    *hdi_post.smoothed_posterior_observed.isel(observed_state=0).values.T,
    alpha=0.25,
    label="94% HDI",
)
ax.plot(airpass.index, airpass.values, label="Data")
ax.legend()
plt.show()

### Forcasting

In [ ]:
forecasts = ss_mod.forecast(idata, start=airpass.index[-1], periods=24)

In [ ]:
fig, ax = plt.subplots()
ax.plot(airpass.index, airpass, label="Data")
hdi = az.hdi(forecasts.to_dataset().forecast_observed.isel(observed_state=0))
ax.fill_between(
    forecasts.coords["time"], *hdi.values.T, color="tab:orange", alpha=0.5, label="94% HDI"
)
ax.plot(
    forecasts.coords["time"],
    forecasts.forecast_observed.mean(dim=["chain", "draw"]).values.squeeze(),
    color="k",
    alpha=1,
    label="Mean Forecast",
)
ax.legend()
plt.show()

## Cherry Blossom Data

One last famous dataset is the Cherry Blossom data. A spline regression fit to this dataset is used on the cover of Richard McElreath's Statistical Rethinking. In the spirit of that cover, we can fit a time-varying spline regression with a smooth trend to the dataset. We can also include serial-correlated errors using an AutoRegressive component.

In [ ]:
blossom_data = pd.read_csv(pm.get_data("cherry_blossoms.csv"), sep=";")
blossom_data.year = pd.date_range(start="01-01-801", periods=1215, freq="YS", unit="s")
blossom_data.set_index("year", inplace=True)

blossom_data = blossom_data[["doy", "temp"]].dropna(subset=["temp"]).iloc[20:]
blossom_data.index.freq = blossom_data.index.inferred_freq

In [ ]:
fig, ax = plt.subplots()
ax2 = ax.twinx()
(doy_line,) = ax.plot(blossom_data.index, blossom_data.doy, label="Day of Year")
(temp_line,) = ax2.plot(blossom_data.index, blossom_data.temp, color="tab:red", label="Temp")

ax.legend(handles=[doy_line, temp_line], loc="upper left")
ax2.grid(visible=False)
ax.set_ylabel("Day of Bloom")
ax2.set_ylabel("Temperature")
plt.show()

In [ ]:
ll = st.LevelTrend(order=2, innovations_order=0)
temp_reg = st.Regression(name="exog", innovations=True, state_names=["Temperature"])
ar_error = st.Autoregressive(name="ar", order=1)
obs_error = st.MeasurementError("obs_error")
mod = (ll + temp_reg + ar_error + obs_error).build()

In [ ]:
with pm.Model(coords=mod.coords) as blossom_mod:
    # Provide data for the exogenous regression model
    exog_data = pm.Data("data_exog", blossom_data.temp.values[:, None], dims=["time", "state_exog"])

    P0_sigma = pm.Gamma("P0_diag", alpha=2, beta=1, shape=(1,))
    P0 = pm.Deterministic("P0", pt.eye(mod.k_states) * P0_sigma, dims=["state", "state_aux"])

    initial_trend = pm.Normal(
        "initial_level_trend", mu=[100, 0], sigma=[30, 1], dims=["state_level_trend"]
    )
    beta_temperature = pm.Normal("beta_exog", mu=0, sigma=10, dims=["state_exog"])
    sigma_beta_exog = pz.maxent(pz.Gamma(), lower=1e-3, upper=0.5, plot=False).to_pymc(
        "sigma_beta_exog", dims=["state_exog"]
    )

    # Keep the prior mass in the stationary region (-1, 1)
    ar_params = pz.maxent(pz.Normal(), lower=-0.9, upper=0.9, plot=False).to_pymc(
        "params_ar", dims=["lag_ar"]
    )

    sigma_error = pz.maxent(pz.Gamma(), lower=0.1, upper=5, plot=False).to_pymc("sigma_obs_error")
    sigma_ar = pz.maxent(pz.Gamma(), lower=0.1, upper=5, plot=False).to_pymc("sigma_ar")

    mod.build_statespace_graph(blossom_data.doy)

In [ ]:
with blossom_mod:
    # Sample normally for comparison
    idata = pm.sample(target_accept=0.9)

In [ ]:
var_names = [
    "initial_level_trend",
    "params_ar",
    "beta_exog",
    "sigma_ar",
    "sigma_beta_exog",
    "sigma_obs_error",
]

az.plot_trace_dist(idata, var_names=var_names)

### Post-Estimation

In [ ]:
post = mod.sample_conditional_posterior(idata, mvn_method="cholesky")
hdi_post = az.hdi(post.to_dataset())

In [ ]:
fig, ax = plt.subplots()

mu = post.predicted_posterior_observed.mean(dim=["chain", "draw"])
ax.plot(blossom_data.index, mu)
ax.fill_between(
    blossom_data.index,
    *hdi_post.smoothed_posterior_observed.isel(observed_state=0).values.T,
    alpha=0.25,
)
blossom_data.doy.plot(ax=ax)

In [ ]:
component_idata = mod.extract_components_from_idata(post)

Evidently, after accounting for the temperature, there is no more trend in the time series. There is some evidence of positive autocorrelation, though.

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(14, 8))
x_values = component_idata.coords["time"]
component_idxs = [0, 1, 2, 3]
for axis, idx, name in zip(fig.axes, component_idxs, ["Level", "Trend", "Temperature", "AR"]):
    data = component_idata.smoothed_posterior.isel(state=idx)
    if name == "Temperature":
        data = data * blossom_data.temp.values[None, None]
    hdi = az.hdi(data)
    mean = data.mean(dim=["chain", "draw"])

    axis.plot(x_values, mean)
    axis.fill_between(x_values, *hdi.values.T, color="tab:blue", alpha=0.1)
    axis.set_title(name)
plt.show()